# Extracting Balanced Motions

In [6]:
import os

import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# import xskillscore
from scipy.interpolate import griddata
from scipy import signal

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy

import scipy.fftpack as fp

# from astropy.convolution import Gaussian2DKernel, interpolate_replace_nans

from dask import delayed,compute

import glob

import gc

import pyinterp.backends.xarray
# Module that handles the filling of undefined values.
import pyinterp.fill

## 1 - Load SSH data 

In [ ]:
dir_input = # Prescribe input dir
pattern = 'MITgcm_'

In [ ]:
ds_ssh = xr.open_mfdataset(os.path.join(dir_input,pattern+'*.nc'))#.sel(time=slice(np.datetime64("2012-04-01T00:00:00.000000000"),np.datetime64("2012-06-30T23:00:00.000000000")))
#ds_ssh = ds_ssh.sel(longitude=slice(210,215),latitude=slice(25,30),time=slice(np.datetime64("2012-04-01T00:00:00.000000000"),np.datetime64("2012-06-30T23:00:00.000000000")))
ds_ssh

In [ ]:
ds_ssh = ds_ssh.chunk({'longitude':5,'latitude':5,'time':ds_ssh.time.size})

In [ ]:
ds_ssh

## 2 - Process parameters 

In [ ]:
longitude = ds_ssh.longitude.values
latitude = ds_ssh.latitude.values
time = ds_ssh.time.values
nt = time.size
ny = latitude.size
nx = longitude.size

This cell calculates the Coriolis frequency and period : 

In [ ]:
# Coriolis period
f = 2*2*np.pi/86164*np.sin(np.mean(np.deg2rad(latitude))) 
T = 2*np.pi/f
print("Coriolis period in hours :",T//3600)

This cell computes the Gaussian Kernel for a low pass filter at a cutting frequency of the Coriolis period. 

In [ ]:
dt = 3600 # number of seconds in a hour 
window_len = int(2*T//dt) # length of the gaussian filter window 
time_window = np.arange(-window_len,window_len+1) # array time steps to compute the kernel 
exp_window = np.exp(-np.square(time_window/(T/dt))) # array of kernel values 

plt.plot(time_window,exp_window)

ntw = time_window.size


## 3 - Filtering for BM extraction 

In [ ]:
weight = xr.DataArray(exp_window, dims=['window'])
ssh_dedac = ds_ssh.ssh_dedac.chunk({'longitude':5*48,'latitude':5*48,'time':ds_ssh.time.size})#[:nt-nt%ntw]
ssh_bm = ssh_dedac.rolling(time=ntw, center=True).construct('window').dot(weight)/weight.sum()
ssh_bm = ssh_bm.chunk({'longitude':5*48,'latitude':5*48,'time':-1})
ssh_bm

In [ ]:
ssh_hf = ssh_dedac - ssh_bm
ssh_hf = ssh_hf.chunk({'longitude':5*48,'latitude':5*48,'time':ds_ssh.time.size})
ssh_hf

In [ ]:
_ssh_bm = ssh_bm.sel(longitude=200,latitude =20).load()
_ssh_bm

In [ ]:
_ssh_bm[1556]

In [ ]:
_ssh_dedac = ssh_dedac.sel(longitude=200,latitude =20)

In [ ]:
def save_bm_hf(_ssh_dedac,_ssh_bm,dir_output):
    _ssh_bm = _ssh_bm.load()
    _ssh_hf = _ssh_dedac - _ssh_bm

    dsout = xr.Dataset({'ssh_dedac':(('time','latitude','longitude'),_ssh_dedac.data),
                    'ssh_bm':(('time','latitude','longitude'),_ssh_bm.data),
                    'ssh_hf':(('time','latitude','longitude'),_ssh_hf.data)
                    },
                    coords={'time':('time',_ssh_dedac.time.values),'latitude':('latitude',_ssh_dedac.latitude.values),'longitude':('longitude',_ssh_dedac.longitude.values)}
                    )

    name_file = "MITgcm_filt_"+str(_ssh_dedac.latitude.values[0])+"_"+str(_ssh_dedac.longitude.values[0])+".nc"

    #dsout.to_netcdf(os.path.join(dir_output,name_file))

    del _ssh_dedac, _ssh_bm 
    gc.collect()

In [ ]:
dir_output = "/bettik/bellemva/MITgcm/MITgcm_filtered"


In [ ]:
save_bm_hf(ssh_dedac.sel(latitude = slice(10,12),longitude = slice(180,182)),
               ssh_bm.sel(latitude = slice(10,12),longitude = slice(180,182)),
               dir_output)

In [ ]:
new_file = xr.open_dataset("/bettik/bellemva/MITgcm/MITgcm_filtered/MITgcm_filt_10.0_180.0.nc")

In [ ]:
new_file

In [ ]:
new_file.ssh_bm[51].plot()

In [ ]:
delayed_results = []
for i in range(2):
    for j in range (2) : 
        res = delayed(save_bm_hf)(x[i],y[j])
        delayed_results.append(res)
results = compute(*delayed_results, scheduler="processes")

In [ ]:
ssh_bm = ssh_bm.load()